# TenderLogic AI -- Enterprise Procurement Audit Platform
**Complete Modern React Web Interface and AI Cross-Examination Engine on Google Colab**

---

### How to Launch:
1. **(Optional)** Add your `GROQ_API_KEY` in Google Colab **Secrets** (the key icon on the left sidebar), or enter it when prompted in Step 2.
2. Click **Runtime** -> **Run all** (or press `Ctrl + F9`).
3. Wait for the final cell to display your **Public Cloudflare URL** (e.g. `https://....trycloudflare.com`).
4. Click the link to open the exact, modern React web application with dark-mode UI, document intake cards, audit journey stepper, and results board!

In [1]:
# Step 1: Install Dependencies and Cloudflare Tunnel
print('Installing Python dependencies...')
!pip install -q fastapi uvicorn python-multipart groq pymupdf rank_bm25 pydantic pytesseract pillow python-dotenv
!apt-get install -y -qq tesseract-ocr > /dev/null 2>&1

print('Installing Cloudflare Tunnel binary...')
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print('Environment ready!')


Installing Python dependencies...
Installing Cloudflare Tunnel binary...
Environment ready!


In [2]:
# Step 2: Configure Groq API Key
import os

try:
    from google.colab import userdata
    secret_key = userdata.get('GROQ_API_KEY')
    if secret_key:
        os.environ['GROQ_API_KEY'] = secret_key
        print('Loaded GROQ_API_KEY from Google Colab Secrets!')
except Exception:
    pass

if not os.environ.get('GROQ_API_KEY'):
    from getpass import getpass
    print('Tip: Free API keys available at https://console.groq.com')
    key_input = getpass('Enter your Groq API Key (starts with gsk_...) or press Enter for offline mode: ')
    if key_input.strip():
        os.environ['GROQ_API_KEY'] = key_input.strip()
        print('GROQ_API_KEY saved!')
    else:
        print('Operating in offline rule-based heuristic mode.')


Loaded GROQ_API_KEY from Google Colab Secrets!


In [3]:
# Step 3: Unpack Modern React Frontend and Backend Application
import os
import io
import base64
import zipfile

BUNDLE_B64 = '''...'''

print('Extracting TenderLogic AI application files...')
zip_bytes = base64.b64decode(BUNDLE_B64)
with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as z:
    z.extractall('.')

print('TenderLogic AI components verified:')
print('  [x] backend/server.py (FastAPI)')
print('  [x] frontend/dist/ (Complete React Production Application)')
print('  [x] pipeline.py (Document parsing and OCR)')
print('  [x] ai_engine.py (Groq LLM and clause retriever)')
print('  [x] main.py (Server entry point)')


Extracting TenderLogic AI application files...
TenderLogic AI components verified:
  [x] backend/server.py (FastAPI)
  [x] frontend/dist/ (Complete React Production Application)
  [x] pipeline.py (Document parsing and OCR)
  [x] ai_engine.py (Groq LLM and clause retriever)
  [x] main.py (Server entry point)


In [9]:
# Step 4: Launch FastAPI Server and Expose Public Website
import time
import subprocess
import re
from IPython.display import HTML, display

# Terminate any process on port 8000
!fuser -k 8000/tcp > /dev/null 2>&1

print('Launching FastAPI backend server...')
server_log = open('server_output.log', 'w')
server_proc = subprocess.Popen(
    ['python', 'main.py', '--port', '8000'],
    stdout=server_log,
    stderr=server_log,
    env=os.environ
)

time.sleep(3)

print('Starting Cloudflare Tunnel to expose the React web app...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--protocol', 'http2'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
for _ in range(80):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('\n' + '=' * 65)
    print('  TENDERLOGIC AI IS ONLINE!')
    print('=' * 65)
    print(f'\n  👉 CLICK HERE TO OPEN THE WEBSITE:\n  {public_url}\n')
    print('=' * 65)
    card_html = f'''
    <div style="background: linear-gradient(135deg, #090D16, #1C2740); border: 2px solid #6366F1; border-radius: 16px; padding: 24px; text-align: center; margin: 20px 0;">
        <h2 style="color: #FFFFFF; margin: 0 0 8px 0; font-family: sans-serif; font-size: 24px;">TenderLogic AI is Ready</h2>
        <p style="color: #94A3B8; margin: 0 0 18px 0; font-size: 14px;">Your modern React enterprise application is running on Cloudflare Tunnel.</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; background: linear-gradient(90deg, #3B82F6, #6366F1, #8B5CF6); color: white; padding: 14px 32px; border-radius: 12px; font-weight: bold; text-decoration: none; font-size: 16px;">
            ✦ Open TenderLogic AI Website
        </a>
    </div>
    '''
    display(HTML(card_html))
else:
    print('Tunnel startup took longer than expected. Server log:')
    with open('server_output.log') as f:
        print(f.read()[-500:])

Launching FastAPI backend server...
Starting Cloudflare Tunnel to expose the React web app...

  TENDERLOGIC AI IS ONLINE!

  👉 CLICK HERE TO OPEN THE WEBSITE:
  https://long-savannah-psychological-batman.trycloudflare.com



In [5]:
 !cat server_output.log

INFO:     Started server process [3948]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [6]:
!ps aux | grep cloudflared

root        3972  0.1  0.2 1294808 39244 ?       Sl   07:05   0:00 cloudflared tunnel --url http://127.0.0.1:8000
root        4677  0.0  0.0   7344  3612 ?        S    07:07   0:00 /bin/bash -c ps aux | grep cloudflared
root        4679  0.0  0.0   6548  2384 ?        S    07:07   0:00 grep cloudflared
